In [1]:
%pip install -U transformers datasets accelerate peft trl bitsandbytes sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 22.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 120.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 74.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 15.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 16.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 16.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 80.8 MB/s  0:00:006m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 65.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 159.5 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 20.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 131.3

### 0. 허깅페이스 로그인

In [2]:
import os
print(os.getenv("HF_TOKEN") is not None)

True


### 1. 기본 import

In [3]:
import os
import json
import random
import shutil
from collections import Counter

import torch
from datasets import Dataset
from torch.nn.utils.rnn import pad_sequence

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    AutoPeftModelForCausalLM,
)

### 2. 기본 설정

In [4]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "google/gemma-3-12b-it"
OUTPUT_DIR = "./gemma3_12b_f1_final"
HUB_MODEL_ID = "YHPark0208/SKN24_3rd_2Team"

FILE_PATHS = [
    "./data/finetune/regulations_merged_for_finetuning.jsonl",
    "./data/finetune/regulations_history_glossary_final.jsonl",
    "./data/finetune/flag_tire_intro_wiki.jsonl",
    "./data/finetune/jolpica.jsonl",
]

TRAIN_RATIO = 0.9
VALID_RATIO = 0.05
TEST_RATIO = 0.05

MAX_LENGTH = 1024
NUM_EVAL_SAMPLES = 30

### 3. system prompt

In [5]:
SYSTEM_MSG = """
# 절대 규칙 (반드시 준수)
- context에 없는 숫자(그리드 페널티, 타이어 세트 수, 시간 제한 등)는 절대 생성 금지. context에 숫자가 없으면 '현재 데이터베이스에서 찾을 수 없습니다'로 답변
- context 외의 지식으로 보완하거나 재해석 금지. 근거 없는 답변 금지
- 규정 문서와 기타 문서 충돌 시 규정 문서 우선. 2026 규정 기준으로 답변

# SYSTEM RULE
-역할: F1 스포츠 경기 전문 질의응답 시스템
-조건: 사용자가 입문자라고 가정, 쉽고 간단하게 경기 규칙, 용어, 현상을 설명
-제한: context의 내용을 그대로 해석할 것.
 context에 명시된 조항 번호와 내용을 우선으로 하고,
 유사한 답변을 찾을 수 없는 경우, '현재 데이터베이스에서 찾을 수 없습니다.' 답변
-언어: 사용자는 한국어 사용자 → 출력은 한국어 구어체 답변
-사용자의 한국어 구어체의 맥락을 적절히 이해 필요
-친절한 말투로 답변
-F1 기술 용어는 원문 그대로 사용하거나 혹은 정확하게 번역
"""

### 4. 데이터 로더

In [ ]:
# 텍스트에서 제거할 불필요한 패턴 목록
BAD_PATTERNS = [
    "<|file_separator|>",
    "</div>",
    "<div>",
]

# clean_text: 텍스트에서 불필요한 패턴 제거 및 양쪽 공백 제거
def clean_text(text):
    text = str(text).strip()
    for pat in BAD_PATTERNS:
        text = text.replace(pat, "")
    return text.strip()


# normalize_messages: 메시지 리스트를 정제하여 모델 입력에 적합한 형태로 변환
def normalize_messages(messages, system_msg=SYSTEM_MSG):
    cleaned = []

    for msg in messages:
        # msg.get : 딕셔너리에서 "role"과 "content" 키의 값을 가져오며, 키가 없을 경우 기본값을 반환
        role = str(msg.get("role", "")).strip().lower()
        content = msg.get("content", "")

        # isinstance : content가 리스트인지 확인. 리스트인 경우 각 항목이 딕셔너리이고 "type"이 "text"인 경우 해당 텍스트를 추출하여
        # parts 리스트에 추가
        if isinstance(content, list):
            parts = []
            for item in content:
                if isinstance(item, dict) and item.get("type") == "text":
                    # item.get("text", ""): item 딕셔너리에서 "text" 키의 값을 가져오며, 키가 없을 경우 빈 문자열을 반환
                    parts.append(item.get("text", ""))
            content = "\n".join(parts)

        content = clean_text(content)

        if role not in {"system", "user", "assistant"}:
            continue
        if not content:
            continue

        cleaned.append({"role": role, "content": content})

    # cleaned 메시지 리스트에 "user"와 "assistant" 역할이 모두 포함되어 있는지 확인. 둘 중 하나라도 없으면 None 반환
    if not cleaned:
        return None

    # any : cleaned 리스트에서 메시지의 "role"이 "user"인 항목이 하나라도 있는지 확인. 마찬가지로 "assistant" 역할도 확인
    has_user = any(m["role"] == "user" for m in cleaned)
    has_assistant = any(m["role"] == "assistant" for m in cleaned)
    if not has_user or not has_assistant:
        return None

    if cleaned[0]["role"] != "system":
        cleaned = [{"role": "system", "content": system_msg}] + cleaned
    else:
        cleaned[0]["content"] = system_msg

    if cleaned[-1]["role"] != "assistant":
        return None

    return cleaned

# load_jsonl_chat_files: 지정된 JSONL 파일 경로에서 채팅 데이터를 로드하여 정제된 메시지 리스트로 반환
def load_jsonl_chat_files(file_paths):
    data = []
    stats = Counter()

    for path in file_paths:
        # basename : 파일 경로에서 파일 이름만 추출하여 source 변수에 저장
        source = os.path.basename(path)

        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue

                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    continue

                if "messages" in obj:
                    messages = obj["messages"]
                elif "user" in obj and "assistant" in obj:
                    messages = []
                    if "system" in obj and str(obj["system"]).strip():
                        messages.append({"role": "system", "content": obj["system"]})
                    messages.append({"role": "user", "content": obj["user"]})
                    messages.append({"role": "assistant", "content": obj["assistant"]})
                else:
                    continue

                messages = normalize_messages(messages)
                if messages is None:
                    continue

                data.append({
                    "source": source,
                    "messages": messages,
                })
                stats[source] += 1

    print("총 샘플 수:", len(data))
    for k, v in stats.items():
        print(f"- {k}: {v}")

    return data

# make_dedupe_key: 메시지 리스트를 하나의 문자열로 변환하여 중복 제거에 사용할 키 생성
def make_dedupe_key(example):
    return "\n".join([f"{m['role']}::{m['content']}" for m in example["messages"]])

# dedupe_dataset: 데이터셋에서 중복된 샘플을 제거하여 고유한 샘플만 남긴 새로운 리스트 반환
def dedupe_dataset(dataset):
    seen = set()
    deduped = []
    for ex in dataset:
        key = make_dedupe_key(ex)
        if key in seen:
            continue
        seen.add(key)
        deduped.append(ex)
    print("중복 제거 후:", len(deduped))
    return deduped

### 5. 데이터 로드 / 분할

In [7]:
dataset = load_jsonl_chat_files(FILE_PATHS)
dataset = dedupe_dataset(dataset)

random.shuffle(dataset)

n = len(dataset)
train_end = int(n * TRAIN_RATIO)
valid_end = int(n * (TRAIN_RATIO + VALID_RATIO))

train_data = dataset[:train_end]
valid_data = dataset[train_end:valid_end]
test_data = dataset[valid_end:]

print("train:", len(train_data))
print("valid:", len(valid_data))
print("test :", len(test_data))

train_dataset = Dataset.from_list(train_data)
valid_dataset = Dataset.from_list(valid_data)
test_dataset = Dataset.from_list(test_data)

총 샘플 수: 15260
- regulations_merged_for_finetuning.jsonl: 8512
- regulations_history_glossary_final.jsonl: 1570
- flag_tire_intro_wiki.jsonl: 1449
- jolpica.jsonl: 3729
중복 제거 후: 15219
train: 13697
valid: 761
test : 761


### 6. dtype / quantization

In [ ]:
# get_torch_dtype: 현재 시스템에서 사용할 수 있는 최적의 torch dtype 결정. 
# CUDA 지원 여부와 GPU 아키텍처에 따라 bfloat16, float16, 또는 float32 반환
def get_torch_dtype():
    if not torch.cuda.is_available():
        return torch.float32, False, False

    # get_device_capability : 현재 CUDA 장치의 아키텍처 버전을 반환. major는 주요 버전 번호, minor는 부 버전 번호를 나타냄
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        return torch.bfloat16, True, False
    return torch.float16, False, True

TORCH_DTYPE, USE_BF16, USE_FP16 = get_torch_dtype()
print("dtype:", TORCH_DTYPE, "bf16:", USE_BF16, "fp16:", USE_FP16)

# BitsAndBytesConfig: 4비트 양자화 설정을 정의하는 클래스. 
# 모델을 4비트로 로드하여 메모리 사용량을 줄이고, 계산 효율성을 높이는 데 사용
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=TORCH_DTYPE,
)

dtype: torch.bfloat16 bf16: True fp16: False


### 7. 토크나이저 / 모델

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# pad_token이 없는 경우, eos_token을 pad_token으로 설정하여 패딩 토큰으로 사용. 패딩은 오른쪽에서 적용
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    torch_dtype=TORCH_DTYPE,
    device_map="auto",
)

base_model.config.use_cache = False
# prepare_model_for_kbit_training: 4비트 양자화된 모델을 LoRA 훈련에 적합한 형태로 준비.
base_model = prepare_model_for_kbit_training(base_model)

config.json:   0%|          | 0.00/916 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

### 8. LoRA

In [ ]:
lora_config = LoraConfig(
    # r = 16: LoRA의 랭크를 16으로 설정. 
    # 랭크는 LoRA가 학습하는 저차원 표현의 크기를 나타내며, 모델의 적응 능력과 훈련 효율성에 영향을 미침
    r=16,
    # lora_alpha=32: LoRA의 스케일링 팩터를 32로 설정. 
    # 이 값은 LoRA가 학습하는 저차원 표현의 업데이트 크기를 조절하는 역할을 하며, 모델의 적응 능력과 훈련 안정성에 영향을 미침
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 24,219,648 || all params: 12,211,544,688 || trainable%: 0.1983


### 9. chat template / collator

In [ ]:
# render_chat: 메시지 리스트를 모델 입력에 적합한 텍스트 형식으로 변환.
def render_chat(messages, add_generation_prompt=False):
    # apply_chat_template: 메시지 리스트를 모델이 이해할 수 있는 텍스트 형식으로 변환하는 함수. 
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=add_generation_prompt
    )

# data_collator: 모델 입력으로 사용할 배치를 생성하는 함수.
def data_collator(features):
    input_id_list = []
    attention_mask_list = []
    token_type_id_list = []
    label_list = []

    for feature in features:
        messages = feature["messages"]

        prompt_messages = messages[:-1]
        full_messages = messages

        # render_chat: 메시지 리스트를 모델 입력에 적합한 텍스트 형식으로 변환.
        full_text = render_chat(full_messages, add_generation_prompt=False)
        prompt_text = render_chat(prompt_messages, add_generation_prompt=True)

        full_enc = tokenizer(
            full_text,
            add_special_tokens=False,
            return_attention_mask=True,
            return_token_type_ids=True,
        )
        prompt_enc = tokenizer(
            prompt_text,
            add_special_tokens=False,
            return_attention_mask=True,
            return_token_type_ids=True,
        )

        full_ids = full_enc["input_ids"]
        full_token_type_ids = full_enc.get("token_type_ids", [0] * len(full_ids))
        prompt_ids = prompt_enc["input_ids"]

        labels = full_ids.copy()
        prompt_len = min(len(prompt_ids), len(labels))
        labels[:prompt_len] = [-100] * prompt_len

        full_ids = full_ids[-MAX_LENGTH:]
        full_token_type_ids = full_token_type_ids[-MAX_LENGTH:]
        labels = labels[-MAX_LENGTH:]
        attention_mask = [1] * len(full_ids)

        input_id_list.append(torch.tensor(full_ids, dtype=torch.long))
        attention_mask_list.append(torch.tensor(attention_mask, dtype=torch.long))
        token_type_id_list.append(torch.tensor(full_token_type_ids, dtype=torch.long))
        label_list.append(torch.tensor(labels, dtype=torch.long))



    # pad_sequence: 시퀀스 리스트를 패딩하여 동일한 길이로 만들어주는 함수. 
    # batch_first=True로 설정하여 배치 차원이 첫 번째가 되도록 함.
    batch_input_ids = pad_sequence(
        input_id_list,
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )
    batch_attention_mask = pad_sequence(
        attention_mask_list,
        batch_first=True,
        padding_value=0
    )
    batch_token_type_ids = pad_sequence(
        token_type_id_list,
        batch_first=True,
        padding_value=0
    )
    batch_labels = pad_sequence(
        label_list,
        batch_first=True,
        padding_value=-100
    )

    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_mask,
        "token_type_ids": batch_token_type_ids,
        "labels": batch_labels,
    }

### 10. 학습 설정

In [12]:
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=20,
    eval_steps=300,
    save_steps=300,
    save_total_limit=3,
    eval_strategy="steps",
    save_strategy="steps",
    bf16=USE_BF16,
    fp16=USE_FP16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    report_to="none",
    remove_unused_columns=False,
    seed=SEED,
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
    hub_strategy="every_save",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


### 11. trainer / 학습

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
)

trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
300,1.207456,1.318689
600,1.393669,1.251326
900,1.227251,1.199565
1200,1.285448,1.172766
1500,1.134784,1.145866
1800,0.960177,1.129542
2100,1.016106,1.107419
2400,0.934741,1.097512
2700,0.868000,1.083794
3000,0.865545,1.078409


TrainOutput(global_step=3426, training_loss=1.0751628135284763, metrics={'train_runtime': 27133.0542, 'train_samples_per_second': 1.01, 'train_steps_per_second': 0.126, 'total_flos': 6.268127156618454e+17, 'train_loss': 1.0751628135284763, 'epoch': 2.0})

### 12. 저장 / 업로드

In [19]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

trainer.push_to_hub()
tokenizer.push_to_hub(HUB_MODEL_ID)

print("학습/업로드 완료")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


학습/업로드 완료


### 13. 파인튜닝 모델 다시 로드

In [20]:
finetuned_model = AutoPeftModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=TORCH_DTYPE,
    device_map="auto",
)

finetuned_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

### 14. 추론 함수

In [21]:
def build_generation_prompt(messages):
    prompt_messages = messages[:-1] if messages[-1]["role"] == "assistant" else messages
    return finetuned_tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

def generate_text(model, tokenizer, prompt, max_new_tokens=64):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

### 15. 자동 평가 저장

In [1]:
eval_samples = random.sample(test_data, min(NUM_EVAL_SAMPLES, len(test_data)))

results = []

for sample in eval_samples:
    user_msgs = [m["content"] for m in sample["messages"] if m["role"] == "user"]
    question = user_msgs[-1] if user_msgs else "(질문 없음)"
    label = sample["messages"][-1]["content"]
    source = sample.get("source", "unknown")

    pred = generate_text(
        finetuned_model,
        finetuned_tokenizer,
        build_generation_prompt(sample["messages"]),
        max_new_tokens=64
    )

    results.append({
        "source": source,
        "question": question,
        "label": label,
        "prediction": pred,
    })

with open("final_eval_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("저장 완료: final_eval_results.json")

NameError: name 'random' is not defined

### 16. 평가 일부 출력

In [2]:
for i, row in enumerate(results[:10], start=1):
    print(f"\n[Sample {i}]")
    print("[Source]", row["source"])
    print("[질문]", row["question"])
    print("[정답]", row["label"])
    print("[예측]", row["prediction"])
    print("=" * 120)

NameError: name 'results' is not defined